# Feast feature store — quickstart

Companion notebook to
[`docs/tools/feast/README.md`](../../docs/tools/feast/README.md). Walks
through the core Feast loop end to end: define features once, retrieve a
point-in-time-correct training set from the **offline store**, train a
model, push features into the **online store**, and serve a low-latency
prediction from it — the same features, two different retrieval paths.

Uses the bundled `feature_repo/` in this project (generated by `feast
init` and trimmed down to the essentials — see
[`feature_repo/feature_definitions.py`](feature_repo/feature_definitions.py)).
The sample data is Feast's own driver-stats dataset: hourly
`conv_rate`/`acc_rate`/`avg_daily_trips` for 5 drivers, generated fresh up
to today whenever `feast init` runs.

Setup (uv-managed, see this project's README):

```bash
cd projects/feast-demo
uv sync
uv run jupyter notebook feast_quickstart.ipynb
```

In [1]:
import subprocess
from datetime import datetime, timezone

import pandas as pd
from feast import FeatureStore
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

REPO_PATH = "feature_repo"

## 1. Register the feature definitions (`feast apply`)

`feature_repo/feature_definitions.py` declares, in code:

- an **entity** (`driver`, keyed on `driver_id`) — the thing features are
  attached to
- a **feature view** (`driver_hourly_stats`) — which columns, from which
  source file, count as features
- a **feature service** (`driver_activity_v1`) — a named bundle of
  features for a model version to request as a unit

`feast apply` reads that file and writes the registry (`feature_repo/data/registry.db`
— just metadata, no feature values yet) and provisions the online store's
backing table. This is the step that makes feature definitions a shared,
queryable asset instead of scattered feature-engineering code.

In [2]:
result = subprocess.run(["feast", "apply"], cwd=REPO_PATH, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

store = FeatureStore(repo_path=REPO_PATH)
[fv.name for fv in store.list_feature_views()]

No project found in the repository. Using project name feast_demo defined in feature_store.yaml
Applying changes for project feast_demo
Created project feast_demo
Created entity driver
Created feature view driver_hourly_stats
Created feature service driver_activity_v1

Created sqlite table feast_demo_driver_hourly_stats





['driver_hourly_stats']

## 2. Historical features for training (point-in-time correct)

The whole reason a feature store exists rather than just joining tables
yourself: `get_historical_features` takes an `entity_df` of
`(driver_id, event_timestamp)` pairs and, for **each row**, returns the
feature values *as they were at that exact timestamp* — not the latest
values. That's what prevents label leakage when the labels you're
training on came from different points in time.

Here `entity_df` is built by sampling real `(driver_id, event_timestamp)`
pairs straight out of the underlying parquet file, standing in for "here's
when each labeled training example actually happened."

In [3]:
raw = pd.read_parquet(f"{REPO_PATH}/data/driver_stats.parquet")
entity_df = raw[["driver_id", "event_timestamp"]].sample(n=300, random_state=42).reset_index(drop=True)

training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "driver_hourly_stats:conv_rate",
        "driver_hourly_stats:acc_rate",
        "driver_hourly_stats:avg_daily_trips",
    ],
).to_df()

training_df.head()

,driver_id,event_timestamp,conv_rate,acc_rate,avg_daily_trips
0,1004,2021-04-12 07:00:00+00:00,0.001443,0.916711,722
1,1002,2026-07-20 23:00:00+00:00,0.423489,0.414616,417
2,1004,2026-07-21 01:00:00+00:00,0.658915,0.321368,779
3,1002,2026-07-21 01:00:00+00:00,0.214890,0.007125,905
4,1004,2026-07-21 02:00:00+00:00,0.953499,0.316753,562


## 3. Train a model on the retrieved features

A synthetic label (`is_high_converter`) is derived from `conv_rate` just to
have something to train against; the model is deliberately trained on
`acc_rate` and `avg_daily_trips` only (not `conv_rate` itself) to avoid
trivially leaking the label into the features. The point of this section
isn't model quality — it's that the training data came entirely through
Feast's retrieval API, not a hand-rolled join.

In [4]:
training_df["is_high_converter"] = (
    training_df["conv_rate"] > training_df["conv_rate"].median()
).astype(int)

feature_columns = ["acc_rate", "avg_daily_trips"]
X = training_df[feature_columns]
y = training_df["is_high_converter"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
model = LogisticRegression().fit(X_train, y_train)
print("test accuracy:", accuracy_score(y_test, model.predict(X_test)))

test accuracy: 0.4666666666666667


## 4. Materialize features into the online store

`materialize_incremental` copies the *latest* feature values (per entity)
from the offline source into the online store (SQLite here; Redis/DynamoDB
in production) — this is the step that makes low-latency point lookups by
`driver_id` possible at all. Offline retrieval above scans the whole
parquet file per query; online retrieval below is a key-value lookup.

In [5]:
store.materialize_incremental(end_date=datetime.now(timezone.utc))

Materializing 1 feature views to 2026-08-04 16:02:49+00:00 into the sqlite online store.

driver_hourly_stats from 2016-08-06 16:02:49+00:00 to 2026-08-04 16:02:49+00:00:


## 5. Online features for serving

Same feature names, same feature view — but now fetched by entity key with
no timestamp involved, the way a live prediction request would. This is
the same feature *definition* used for training in step 2, so the
training/serving skew that a hand-rolled setup risks (recomputing
"average daily trips" slightly differently in the training pipeline vs.
the serving code) isn't possible here — there's only one definition.

In [6]:
online_features = store.get_online_features(
    features=[
        "driver_hourly_stats:conv_rate",
        "driver_hourly_stats:acc_rate",
        "driver_hourly_stats:avg_daily_trips",
    ],
    entity_rows=[{"driver_id": 1001}, {"driver_id": 1002}, {"driver_id": 1003}],
).to_df()

online_features["predicted_high_converter"] = model.predict(
    online_features[feature_columns]
)
online_features

,driver_id,acc_rate,avg_daily_trips,conv_rate,predicted_high_converter
0,1001,0.995651,553,0.624261,1
1,1002,0.006426,421,0.681352,0
2,1003,0.401848,403,0.073282,0


**Reading this:** that `.predict()` call is what a real-time serving
endpoint would do on every request — look up features for the entity in
the request (by `driver_id`), feed them to the model, return a prediction.
The lookup latency here is what matters in production (SQLite is fine for
a demo; Redis/DynamoDB are the point-lookup backends actually used at
scale).

## 6. Feature services — requesting a named bundle instead of a column list

Instead of spelling out `"driver_hourly_stats:conv_rate"` etc. every time,
a `FeatureService` bundles a specific set of features under one name
(`driver_activity_v1`, defined in `feature_definitions.py`) — the thing an
actual model version would request, so the feature list a model depends on
is versioned and named rather than copy-pasted at every call site.

In [7]:
feature_service = store.get_feature_service("driver_activity_v1")

store.get_online_features(
    features=feature_service,
    entity_rows=[{"driver_id": 1001}],
).to_df()

,driver_id,acc_rate,avg_daily_trips,conv_rate
0,1001,0.995651,553,0.624261


## Summary

| Step | Feast API | Purpose |
|---|---|---|
| Define features | `Entity`, `FeatureView`, `FeatureService` in code | One versioned definition, not scattered feature-engineering logic |
| Register | `feast apply` | Writes the registry; provisions online store tables |
| Training data | `get_historical_features` | Point-in-time-correct join against the offline store — no label leakage |
| Load online store | `materialize_incremental` | Copies latest values into the low-latency online store |
| Serving | `get_online_features` | Key-value lookup by entity, same feature definitions as training |

Related docs: [`docs/tools/feast/README.md`](../../docs/tools/feast/README.md)
— full write-up, alternatives (including Databricks Feature Store),
relationship with MLflow and Evidently.